# Method: Multilevel Perceptron

### I. Imports and Reads

#### Imports

In [655]:
import numpy as np
import csv 

#### Reads

In [656]:
with open('../dataset/processed_test.csv', 'r') as f:
    read = csv.reader(f)
    data = []
    output = []
    for x in read:
        data.append(x[:-1])
        output.append(x[-1])

    test_in = np.array(data, dtype=np.float64)
    test_out = np.array(output, dtype=np.float64)

with open('../dataset/processed_tune.csv', 'r') as f:
    read = csv.reader(f)
    data = []
    output = []
    for x in read:
        data.append(x[:-1])
        output.append(x[-1])

    tune_in = np.array(data, dtype=np.float64)
    tune_out = np.array(output, dtype=np.float64)

with open('../dataset/processed_train.csv', 'r') as f:
    read = csv.reader(f)
    data = []
    output = []
    for x in read:
        data.append(x[:-1])
        output.append(x[-1])

    train_in = np.array(data, dtype=np.float64)
    train_out = np.array(output, dtype=np.float64)

### II. Setup

#### Initial Parameters


In [657]:
learning_rate = 0.0001
input_size = train_in.shape[1]
N = train_in.shape[0]
batch_size = N
layer_sizes = [16, 16, 16] + [1]


#### Activation Functions

In [658]:
def ReLU(layers):
    if type(layers) == list:
        layers = np.array(layers, dtype=np.float64)
    return np.where(layers < 0, 0, layers)

def ReLUDerivative(layers):
    return np.where(layers < 0, 0, 1)

#### Layer Setup

In [659]:
weights = [np.zeros((input_size, layer_sizes[0]), dtype=np.float64)]
biases = []
layers = [np.zeros((batch_size, input_size), dtype=np.float64)]
for x in layer_sizes:
    biases.append(np.ones((x), dtype=np.float64))
    layers.append(np.zeros((batch_size, x), dtype=np.float64))

for i in range(0, len(layer_sizes)-1):
    weights.append(np.zeros((layer_sizes[i], layer_sizes[i+1]), dtype=np.float64))


print([x.shape for x in layers])
print([x.shape for x in weights])
print([x.shape for x in biases])

[(3174, 6), (3174, 16), (3174, 16), (3174, 16), (3174, 1)]
[(6, 16), (16, 16), (16, 16), (16, 1)]
[(16,), (16,), (16,), (1,)]


### III. Training

In [ ]:
iters = 100000
learning_rate = 0.00001
for iter in range(iters):

    # get batch_size datapoints
    cost = np.zeros((batch_size))
    indexes = np.random.choice(range(N), size = batch_size, replace= False)
    layers[0] = train_in[indexes]
    expected = train_out[indexes].reshape(-1, 1)

    #forward prop
    for i in range(len(layers)-1):
        if i < len(layers)-1: 
            layers[i+1] = ReLU(layers[i].dot(weights[i])+biases[i])
        else:
            layers[i+1] = layers[i].dot(weights[i])+biases[i]

    cost = (layers[-1] - expected)**2

    # print(cost)
    print(f"Mean Squared Error [{iter}]: {cost.mean()}")
    # backward prop

    bias_diff = [np.zeros_like(x) for x in biases]
    weights_diff = [np.zeros_like(x) for x in weights]

    delta = 2 * (layers[-1] - expected)
    for i in range(len(layer_sizes) - 1, -1, -1):
        weights_diff[i] = layers[i].T @ delta / batch_size
        bias_diff[i] = delta.mean(axis = 0)

        if i > 0:
            delta = delta @ weights[i].T * ReLUDerivative(layers[i])

    for x in range(len(layer_sizes)-1, -1, -1):
            weights[x] = weights[x] - learning_rate * weights_diff[x]
            biases[x] = biases[x] - learning_rate * bias_diff[x]


Mean Squared Error [0]: 1662.562158979206
Mean Squared Error [1]: 1651.4937137836314
Mean Squared Error [2]: 1640.4775408997268
Mean Squared Error [3]: 1629.4684108209597
Mean Squared Error [4]: 1618.4225912612546
Mean Squared Error [5]: 1607.2976417311943
Mean Squared Error [6]: 1596.0522243672406
Mean Squared Error [7]: 1584.6459246361417
Mean Squared Error [8]: 1573.0390762019804
Mean Squared Error [9]: 1561.1925847447508
Mean Squared Error [10]: 1549.0677458576179
Mean Squared Error [11]: 1536.62605224949
Mean Squared Error [12]: 1523.8289851192262
Mean Squared Error [13]: 1510.6377832661392
Mean Squared Error [14]: 1497.0131803383292
Mean Squared Error [15]: 1482.9150939703102
Mean Squared Error [16]: 1468.3022376930708
Mean Squared Error [17]: 1453.131602953762
Mean Squared Error [18]: 1437.3577171382497
Mean Squared Error [19]: 1420.931512321812
Mean Squared Error [20]: 1403.798518644957
Mean Squared Error [21]: 1385.895890760067
Mean Squared Error [22]: 1367.147421807685
Mean S

KeyboardInterrupt: 

### IV. Test Weights on a Dataset

In [ ]:
layers[0] = test_in
expected = test_out.reshape(-1, 1)

#forward prop
for i in range(len(layers)-1):
    if i < len(layers)-1: 
        layers[i+1] = ReLU(layers[i].dot(weights[i])+biases[i])
    else:
        layers[i+1] = layers[i].dot(weights[i])+biases[i]

cost = (layers[-1] - expected)**2


SStotal = (expected - expected.mean())**2
SStotal = SStotal.sum()

SSres = (layers[-1] - expected)**2
SSres = SSres.sum()

print(f"RMSE [test]: {((layers[-1] - expected)**2).mean() ** 0.5}")
print(f"MAE [test]: {(layers[-1] - expected).mean()}")
print(f"MAPE [test]: {np.abs((expected - layers[-1])/expected).mean()*100}%")
print(f"R2 [test]: {1 - (SSres/SStotal)}")


# get batch_size datapoints
layers[0] = tune_in
expected = tune_out.reshape(-1, 1)

#forward prop
for i in range(len(layers)-1):
    if i < len(layers)-1: 
        layers[i+1] = ReLU(layers[i].dot(weights[i])+biases[i])
    else:
        layers[i+1] = layers[i].dot(weights[i])+biases[i]

cost = (layers[-1] - expected)**2

# print(cost)
SStotal = (expected - expected.mean())**2
SStotal = SStotal.sum()

SSres = (layers[-1] - expected)**2
SSres = SSres.sum()


print(f"\nRMSE [tune]: {((layers[-1] - expected)**2).mean() ** 0.5}")
print(f"MAE [tune]: {(layers[-1] - expected).mean()}")
print(f"MAPE [tune]: {np.abs((expected - layers[-1])/expected).mean()*100}%")
print(f"R2 [tune]: {1 - (SSres/SStotal)}")

RMSE [test]: 2.580145303279937
MAE [test]: 0.22474392518319025
MAPE [test]: 4.260508040729439%
R2 [test]: 0.7203178678447268

RMSE [tune]: 2.1943079465574487
MAE [tune]: 0.9325643404587292
MAPE [tune]: 4.227209317578395%
R2 [tune]: 0.7640268883327577


### V. Save current weights

## DONT FORGET TO RE-COMMENT AFTER SAVING TO PREVENT LOSS OF DATA

In [ ]:
# np.savez("MLP_weights", *weights)
# np.savez("MLP_biases", *biases)